# Week 7 — 完整 Transformer Encoder + MVP v0.3 v1

**目标**：从零实现 4 层 Transformer Encoder + 分类头 + Noam warmup，在合成交易序列上 AUC-PR ≥ 0.95。

**产出**：
- 合成异常数据集（amount-spike / geo-jump / high-freq）
- Sinusoidal PE + Multi-Head Attention + Pre-LN Encoder Block（纯手搓，无 `nn.Transformer`）
- mean pool vs `[CLS]` token 两个变体对比
- Noam schedule 曲线图
- Attention heatmap（异常 vs 正常）
- 与 d2l Ch 11.7 的差异复盘

In [ ]:
# ── Bootstrap (Colab + local) ──────────────────────────────────────
# Env detect → Drive mount → Kaggle creds → reproducibility
import os, sys, pathlib, random

IN_COLAB = 'google.colab' in sys.modules
DRIVE_FOLDER = 'LLM/transformer-anomaly'  # edit if your Drive subfolder differs

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    PROJECT_ROOT = pathlib.Path(f'/content/drive/MyDrive/{DRIVE_FOLDER}')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    for sub in ('data', 'models', 'checkpoints'):
        (PROJECT_ROOT / sub).mkdir(exist_ok=True)
    os.chdir(PROJECT_ROOT)

    # Kaggle credentials via Colab Secrets (left sidebar 🔑 icon).
    # Add KAGGLE_USERNAME and KAGGLE_KEY there; never upload kaggle.json.
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    except Exception as e:
        print(f'[bootstrap] Kaggle secrets not configured: {e}')
else:
    PROJECT_ROOT = pathlib.Path.cwd()

# Reproducibility
import numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[bootstrap] env={"colab" if IN_COLAB else "local"}  device={device}')
print(f'[bootstrap] project_root={PROJECT_ROOT}')

In [ ]:
# 额外依赖
import math
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix)
sns.set_theme(style='whitegrid', palette='Set2')
print('torch', torch.__version__, '| device', device)

## 1. 合成交易数据（小号版 W3）

规格：
- **用户数** N_USERS = 100
- **时长** 7 天，每用户日均 ~8 笔交易
- **特征** 每笔 5 维：`[amount_log, hour_sin, hour_cos, country_id, merchant_id]`
- **异常** 注入 3 类：`amount_spike`（金额突增 10×）、`geo_jump`（国家突变）、`high_freq`（1 分钟连刷）
- **序列** 每用户按时间排序，滑窗 L=32；标签 = 窗口**最后一笔**是否异常

注：只用 100 用户、1 周，目的是让 Colab T4 在 15 min 内跑完端到端；真实数据在 W8 引入。

In [ ]:
# ---- Synthetic transaction generator (self-contained, no shared src/) ----
def gen_synthetic_tx(n_users: int = 100, n_days: int = 7,
                     anomaly_rate: float = 0.02, seed: int = 42) -> pd.DataFrame:
    rng = np.random.RandomState(seed)
    rows = []
    for uid in range(n_users):
        # each user has a "home country" and a preferred merchant cluster
        home_country = rng.randint(0, 10)
        fav_merchants = rng.choice(50, size=5, replace=False)
        # average 8 tx per day -> ~56 per user
        n_tx = int(rng.poisson(8 * n_days))
        timestamps = np.sort(rng.uniform(0, n_days * 86400, size=n_tx))
        for t in timestamps:
            amount = float(np.exp(rng.normal(3.0, 0.8)))  # log-normal around $20
            country = home_country
            merchant = int(rng.choice(fav_merchants))
            rows.append({'user_id': uid, 'ts': float(t),
                         'amount': amount, 'country': country,
                         'merchant': merchant, 'is_anomaly': 0})
    df = pd.DataFrame(rows).sort_values(['user_id', 'ts']).reset_index(drop=True)

    # ---- inject anomalies ----
    n_anom = int(len(df) * anomaly_rate)
    anom_idx = rng.choice(df.index, size=n_anom, replace=False)
    kinds = rng.choice(['amount_spike', 'geo_jump', 'high_freq'], size=n_anom)
    for i, k in zip(anom_idx, kinds):
        if k == 'amount_spike':
            df.at[i, 'amount'] *= rng.uniform(8, 15)
        elif k == 'geo_jump':
            df.at[i, 'country'] = (df.at[i, 'country'] + rng.randint(1, 10)) % 10
        elif k == 'high_freq':
            # pull the timestamp within 60s of the previous row (same user if possible)
            if i > 0 and df.at[i - 1, 'user_id'] == df.at[i, 'user_id']:
                df.at[i, 'ts'] = df.at[i - 1, 'ts'] + rng.uniform(1, 60)
        df.at[i, 'is_anomaly'] = 1
    return df.sort_values(['user_id', 'ts']).reset_index(drop=True)

tx = gen_synthetic_tx()
print(f'rows={len(tx):,}  anomalies={tx.is_anomaly.sum()} ({tx.is_anomaly.mean()*100:.2f}%)')
tx.head()

In [ ]:
# ---- featurize: [amount_log, hour_sin, hour_cos, country_id, merchant_id] ----
def featurize(df: pd.DataFrame) -> np.ndarray:
    amount_log = np.log1p(df['amount'].values).astype(np.float32)
    hour = (df['ts'].values % 86400) / 3600.0  # 0..24
    hour_sin = np.sin(2 * np.pi * hour / 24).astype(np.float32)
    hour_cos = np.cos(2 * np.pi * hour / 24).astype(np.float32)
    country = df['country'].values.astype(np.float32)
    merchant = df['merchant'].values.astype(np.float32)
    # normalize the raw id features to [0, 1] so they play nice with d_model=64
    country /= 10.0
    merchant /= 50.0
    return np.stack([amount_log, hour_sin, hour_cos, country, merchant], axis=1)

# standardize amount_log across the whole dataset (simple z-score)
feats = featurize(tx)
mu, sd = feats[:, 0].mean(), feats[:, 0].std() + 1e-6
feats[:, 0] = (feats[:, 0] - mu) / sd
print('feature shape', feats.shape)

In [ ]:
# ---- build sliding windows per user (L=32). Label = last-row is_anomaly ----
L = 32
def build_sequences(df: pd.DataFrame, feats: np.ndarray, L: int = 32):
    X, y = [], []
    for uid, grp in df.groupby('user_id'):
        idx = grp.index.values
        if len(idx) < L:
            continue
        for end in range(L - 1, len(idx)):
            window = idx[end - L + 1: end + 1]
            X.append(feats[window])
            y.append(df.at[idx[end], 'is_anomaly'])
    return np.stack(X), np.array(y, dtype=np.float32)

X, y = build_sequences(tx, feats, L=L)
print(f'X={X.shape}  y_pos_rate={y.mean()*100:.2f}%')

In [ ]:
# ---- split: 70/15/15 ----
N = len(X)
idx = np.arange(N); np.random.RandomState(0).shuffle(idx)
n_tr, n_va = int(N * 0.70), int(N * 0.15)
tr, va, te = idx[:n_tr], idx[n_tr:n_tr+n_va], idx[n_tr+n_va:]
Xtr, ytr = torch.from_numpy(X[tr]).float(), torch.from_numpy(y[tr]).float()
Xva, yva = torch.from_numpy(X[va]).float(), torch.from_numpy(y[va]).float()
Xte, yte = torch.from_numpy(X[te]).float(), torch.from_numpy(y[te]).float()
print('train', Xtr.shape, 'val', Xva.shape, 'test', Xte.shape)
print('pos rate — train {:.3f} | val {:.3f} | test {:.3f}'.format(
    ytr.mean().item(), yva.mean().item(), yte.mean().item()))

## 2. 从零搭 Transformer Encoder

### 2.1 位置编码（Sinusoidal）

公式（Attention Is All You Need §3.5）：

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}})$$
$$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})$$

关键：这是**不可学习**的，直接加到 token embedding 上。

In [ ]:
class SinusoidalPE(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, d_model)
        return x + self.pe[:, : x.size(1)]

# sanity check — plot first 64 positions
pe = SinusoidalPE(d_model=64, max_len=128).pe.squeeze(0).numpy()
fig, ax = plt.subplots(figsize=(8, 3))
ax.imshow(pe[:64].T, aspect='auto', cmap='RdBu')
ax.set_xlabel('position'); ax.set_ylabel('dim'); ax.set_title('Sinusoidal PE (first 64 positions)')

### 2.2 Multi-Head Self-Attention

scaled dot-product attention: $\text{softmax}(QK^T / \sqrt{d_k}) V$，拆成 h 个头。

为了做 attention 可视化，我们额外返回 `attn` 权重。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        B, L, D = x.shape
        qkv = self.qkv(x).reshape(B, L, 3, self.num_heads, self.d_head)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)  # (3, B, H, L, d_head)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)  # (B, H, L, L)
        attn = F.softmax(attn, dim=-1)
        attn = self.drop(attn)
        out = (attn @ v).transpose(1, 2).reshape(B, L, D)  # (B, L, D)
        out = self.out(out)
        if return_attn:
            return out, attn
        return out

### 2.3 FFN + Pre-LN Encoder Block

Pre-LN 比 Post-LN 稳定：归一化放在子层**输入前**，梯度不需要穿过 LayerNorm 回传，训练收敛更平稳（这是我们选 Pre-LN 的唯一理由）。

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.drop(F.gelu(self.fc1(x))))


class EncoderBlock(nn.Module):
    """Pre-LN Transformer encoder block."""
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.drop1 = nn.Dropout(dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, return_attn: bool = False):
        if return_attn:
            a, attn = self.attn(self.ln1(x), return_attn=True)
            x = x + self.drop1(a)
        else:
            x = x + self.drop1(self.attn(self.ln1(x)))
            attn = None
        x = x + self.drop2(self.ffn(self.ln2(x)))
        return (x, attn) if return_attn else x

### 2.4 完整 Encoder + 分类头（mean pool / `[CLS]` 两版）

- `pool='mean'`：对全 L 个位置的输出取均值，送 Linear
- `pool='cls'`：prepend 一个可学习的 `[CLS]` 向量，取最终第 0 位

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, n_feat: int, d_model: int = 64, num_heads: int = 4,
                 d_ff: int = 128, num_layers: int = 4, dropout: float = 0.1,
                 max_len: int = 64, pool: str = 'mean'):
        super().__init__()
        assert pool in ('mean', 'cls')
        self.pool = pool
        self.input_proj = nn.Linear(n_feat, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model)) if pool == 'cls' else None
        self.pe = SinusoidalPE(d_model, max_len=max_len)
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        # x: (B, L, n_feat)
        h = self.input_proj(x)  # (B, L, D)
        if self.pool == 'cls':
            cls = self.cls_token.expand(h.size(0), -1, -1)
            h = torch.cat([cls, h], dim=1)  # (B, L+1, D)
        h = self.pe(h)
        attns = []
        for blk in self.blocks:
            if return_attn:
                h, a = blk(h, return_attn=True)
                attns.append(a)
            else:
                h = blk(h)
        h = self.ln_f(h)
        pooled = h[:, 0] if self.pool == 'cls' else h.mean(dim=1)
        logit = self.head(pooled).squeeze(-1)
        return (logit, attns) if return_attn else logit

# quick forward smoke test
m = TransformerClassifier(n_feat=X.shape[-1]).to(device)
with torch.no_grad():
    out = m(Xtr[:4].to(device))
print('logit shape', out.shape, '| params', sum(p.numel() for p in m.parameters()))

## 3. Noam Warmup Schedule

$$\text{lr}(step) = d_{model}^{-0.5} \cdot \min(step^{-0.5},\ step \cdot warmup^{-1.5})$$

前 `warmup` 步线性爬升，之后按 $step^{-0.5}$ 衰减。我们在优化器里直接用 `LambdaLR`。

In [ ]:
def noam_lr(step: int, d_model: int, warmup: int) -> float:
    step = max(step, 1)
    return d_model ** -0.5 * min(step ** -0.5, step * warmup ** -1.5)

steps = np.arange(1, 5001)
lrs = [noam_lr(s, 64, 1000) for s in steps]
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(steps, lrs); ax.axvline(1000, c='r', ls='--', alpha=0.5, label='warmup=1000')
ax.set_xlabel('step'); ax.set_ylabel('lr'); ax.set_title('Noam schedule — d_model=64, warmup=1000')
ax.legend()

## 4. 训练循环

- Loss：`BCEWithLogitsLoss(pos_weight=neg/pos)` 对抗不平衡
- Optim：Adam(lr=1.0, betas=(0.9, 0.98), eps=1e-9) + `LambdaLR(noam)` — Noam 是在 lr=1 上乘系数
- Early stopping：val AUC-PR 5 epoch 不涨就停

In [ ]:
def make_loader(X, y, batch_size=64, shuffle=True):
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle)

def eval_model(model, loader):
    model.eval()
    logits, ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits.append(model(xb).cpu().numpy())
            ys.append(yb.numpy())
    logits = np.concatenate(logits); ys = np.concatenate(ys)
    probs = 1 / (1 + np.exp(-logits))
    ap = average_precision_score(ys, probs)
    return ap, probs, ys

def train_loop(model, Xtr, ytr, Xva, yva, epochs=20, batch_size=64,
               warmup=1000, d_model=64, patience=5, tag: str = 'run'):
    pos_w = torch.tensor([(ytr == 0).sum() / max((ytr == 1).sum(), 1)], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lambda s: noam_lr(s + 1, d_model, warmup))
    tr_loader = make_loader(Xtr, ytr, batch_size, shuffle=True)
    va_loader = make_loader(Xva, yva, batch_size=256, shuffle=False)

    best_ap, best_state, bad = -1.0, None, 0
    history = {'train_loss': [], 'val_ap': []}
    for ep in range(1, epochs + 1):
        model.train(); total, n = 0.0, 0
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logit = model(xb)
            loss = loss_fn(logit, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            total += loss.item() * xb.size(0); n += xb.size(0)
        train_loss = total / n
        val_ap, _, _ = eval_model(model, va_loader)
        history['train_loss'].append(train_loss); history['val_ap'].append(val_ap)
        print(f'[{tag}] ep{ep:02d}  loss={train_loss:.4f}  val_AUC-PR={val_ap:.4f}')
        if val_ap > best_ap + 1e-4:
            best_ap, best_state, bad = val_ap, {k: v.detach().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                print(f'[{tag}] early stop at epoch {ep}'); break
    model.load_state_dict(best_state)
    return model, history, best_ap

### 4.1 变体 A — mean pooling

In [ ]:
torch.manual_seed(SEED)
model_mean = TransformerClassifier(n_feat=X.shape[-1], pool='mean', max_len=L + 1).to(device)
model_mean, hist_mean, best_mean = train_loop(
    model_mean, Xtr, ytr, Xva, yva,
    epochs=20, batch_size=64, warmup=1000, d_model=64, tag='mean')
print(f'>> best val AUC-PR (mean) = {best_mean:.4f}')

### 4.2 变体 B — `[CLS]` token

In [ ]:
torch.manual_seed(SEED)
model_cls = TransformerClassifier(n_feat=X.shape[-1], pool='cls', max_len=L + 1).to(device)
model_cls, hist_cls, best_cls = train_loop(
    model_cls, Xtr, ytr, Xva, yva,
    epochs=20, batch_size=64, warmup=1000, d_model=64, tag='cls')
print(f'>> best val AUC-PR (cls) = {best_cls:.4f}')

In [ ]:
# compare learning curves
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].plot(hist_mean['train_loss'], label='mean'); axes[0].plot(hist_cls['train_loss'], label='cls')
axes[0].set_title('train loss'); axes[0].legend()
axes[1].plot(hist_mean['val_ap'], label='mean'); axes[1].plot(hist_cls['val_ap'], label='cls')
axes[1].set_title('val AUC-PR'); axes[1].legend()
print(f'mean={best_mean:.4f}  cls={best_cls:.4f}  winner={"mean" if best_mean >= best_cls else "cls"}')

## 5. 测试集评估

- AUC-PR — 主要指标（极度不平衡下比 ROC 更公允）
- Recall@FPR=0.001 — 业务关心：误报率锁死时能抓到多少欺诈
- 混淆矩阵 — 选一个阈值把概率变预测

In [ ]:
model_best = model_mean if best_mean >= best_cls else model_cls
name_best = 'mean' if best_mean >= best_cls else 'cls'

te_loader = make_loader(Xte, yte, batch_size=256, shuffle=False)
ap, probs, ys = eval_model(model_best, te_loader)
print(f'[{name_best}] TEST AUC-PR = {ap:.4f}')

# Recall@FPR=0.001
fpr, tpr, thr = roc_curve(ys, probs)
mask = fpr <= 0.001
recall_at_fpr = tpr[mask].max() if mask.any() else 0.0
print(f'[{name_best}] TEST Recall@FPR=0.001 = {recall_at_fpr:.4f}')

# pick threshold from val PR curve → max F1
_, p_va, y_va = eval_model(model_best, make_loader(Xva, yva, 256, False))
prec, rec, th = precision_recall_curve(y_va, p_va)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
best_t = th[np.nanargmax(f1[:-1])] if len(th) else 0.5
pred = (probs >= best_t).astype(int)
cm = confusion_matrix(ys, pred)
print(f'threshold (val-tuned F1) = {best_t:.4f}')
print('confusion matrix:\n', cm)

fig, ax = plt.subplots(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['pred 0','pred 1'], yticklabels=['true 0','true 1'])
ax.set_title(f'[{name_best}] confusion matrix')

## 6. 注意力可视化

我们挑 1 条异常 + 1 条正常窗口，看 **layer-4 head-0** 的 attention map（32×32）。

解读技巧：
- 行 = query 位置，列 = key 位置；某行越亮说明该位置在 pool 时更看那列
- 最后一行（预测目标）看向哪几列，常常就是模型认定的"可疑历史"


In [ ]:
def get_attn(model, window: torch.Tensor):
    model.eval()
    with torch.no_grad():
        _, attns = model(window.unsqueeze(0).to(device), return_attn=True)
    # stack -> (num_layers, num_heads, L, L)
    return torch.stack([a.squeeze(0) for a in attns]).cpu().numpy()

pos_idx = (yte == 1).nonzero(as_tuple=True)[0][0].item()
neg_idx = (yte == 0).nonzero(as_tuple=True)[0][0].item()
attn_pos = get_attn(model_best, Xte[pos_idx])
attn_neg = get_attn(model_best, Xte[neg_idx])
print('attn shape per sample:', attn_pos.shape)  # (4, 4, L, L)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(attn_pos[-1, 0], cmap='viridis'); axes[0].set_title('ANOMALY — layer 4 / head 0')
axes[1].imshow(attn_neg[-1, 0], cmap='viridis'); axes[1].set_title('NORMAL — layer 4 / head 0')
for ax in axes:
    ax.set_xlabel('key (history)'); ax.set_ylabel('query')

### 解读（填写自己的观察）

- 异常窗口里，最后一行 attention 是否集中在某几个位置？对应 `tx` 原始数据看是不是 amount_spike / geo_jump 的那几笔？
- 正常窗口 attention 通常比较均匀 / 接近对角，说明模型找不到可疑历史。

> 小练习：换 layer 1 / head 2 再画一次，看早期层 vs 末层的 pattern 差别。

## 7. 复盘 — 我的实现 vs d2l Ch 11.7

逐条对比，填写你实际看到的差异（至少 3 条）：

| # | d2l 11.7 | 本 notebook | 影响 |
|---|----------|------------|------|
| 1 | Post-LN（LayerNorm 在残差**之后**） | Pre-LN（LayerNorm 在子层**之前**） | Pre-LN 训练更稳、对 warmup 依赖更小 |
| 2 | FFN 用 ReLU | 用 GELU | GELU 平滑、在 Transformer 类任务上略好 |
| 3 | d2l 没展示 Noam schedule，而用固定 lr | 本 notebook 加了 Noam LR | warmup 能避免早期大梯度把参数打飞 |
| 4 | d2l 直接返回 encoder 输出做下游任务 | 这里加了 `[CLS]` / mean pool + 分类头 | 分类任务需要把序列聚合成单向量 |
| 5 | *（你的观察）* | *（你的实现）* | *（影响）* |

关键自问：
1. **为什么 Pre-LN 更稳定？** 梯度可以直接沿残差连接回传，不必每层穿过 LayerNorm；在 Post-LN 里深层时梯度会被反复标准化，训练初期经常爆/消失。
2. **Noam schedule 比 cosine 好在哪？** warmup 段显式解决了 "Adam 早期自适应 lr 过大" 的问题；cosine 没有 warmup 时，深层 Transformer 常常在前几百步发散。
3. **mean pool vs `[CLS]`？** 我们这个任务 label 针对最后一笔，mean pool 会"稀释"信号；如果把标签定义改成"窗口里是否含异常"，mean pool 可能更稳。记在笔记里。

---

**本周验收自检**：
- [ ] TEST AUC-PR ≥ 0.95
- [ ] 能口述每个超参的作用
- [ ] 两个 pooling 变体都跑过并对比
- [ ] Noam 曲线 + attention heatmap 都已出图
- [ ] 至少写下 3 条 d2l 差异

下周 W8：用 nanoGPT 风格把这套 Encoder 重构成 `< 150 行` 的紧凑模型，并在 Kaggle 真实数据上验证。